## Load the environment variables

In [1]:
from datasets import load_dataset
from transformers import AutoTokenizer

/home/v/Desktop/Projects/HuggingFace/.env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load a dataset

`load_dataset` downloads the dataset from the Hub and caches it locally.
On subsequent runs it reads from the cache — no re-download needed.

The return value is a `DatasetDict`: a dict-like object with a key per split (`train`, `test`, `validation`).

We'll use **IMDb** — a classic sentiment analysis dataset with 50 000 movie reviews labeled as positive or negative.

In [2]:
ds = load_dataset("imdb")

## Inspect the dataset

Before touching the data it's worth understanding its shape:
- How many rows per split?
- What columns exist and what types do they have?
- What does a single example look like?

Print the `DatasetDict`, the `features`, and the first example of the training split.

In [3]:
print(ds)
print(ds["train"][0])
print(ds["train"].features)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})
{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and

## Filter rows

`.filter()` keeps only the rows where a function returns `True`.
It's lazy-friendly: it iterates in batches without loading everything into RAM.

Filter the training split to keep only **short reviews** (under 500 characters).
Print how many rows remain compared to the original.

In [4]:
short = ds["train"].filter(lambda x: len(x["text"]) < 500)

print(f"Short reviews: {len(short)}/{len(ds['train'])}")
print("-------------------------")
print(short[0])

Short reviews: 2410/25000
-------------------------
{'text': "My interest in Dorothy Stratten caused me to purchase this video. Although it had great actors/actresses, there were just too many subplots going on to retain interest. Plus it just wasn't that interesting. Dialogue was stiff and confusing and the story just flipped around too much to be believable. I was pretty disappointed in what I believe was one of Audrey Hepburn's last movies. I'll always love John Ritter best in slapstick. He was just too pathetic here.", 'label': 0}


## Map a transformation — tokenization

`.map()` applies a function to every row (or batch of rows) and returns a new dataset with the result added as extra columns.

Here we use it to tokenize the `text` column with `distilbert-base-uncased`.
The tokenizer adds two new columns: `input_ids` and `attention_mask`.

Key arguments to `.map()`:
- `batched=True` — passes a batch (default 1000 rows) at a time instead of one row. Much faster.
- `remove_columns` — drop the raw `text` column after tokenizing so the dataset is model-ready.

Tokenizer arguments to use:
- `truncation=True` — clips sequences longer than the model's max length
- `padding="max_length"` — pads shorter sequences so all tensors have the same shape
- `max_length=128`

In [5]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=128)

tokenized = ds.map(tokenize, batched=True, remove_columns=["text"])

Map: 100%|██████████| 50000/50000 [00:04<00:00, 10755.43 examples/s]


## Verify the tokenized dataset

Print the first tokenized example to confirm the new columns are there and the shapes look right.
Check that `input_ids` has exactly 128 elements and that `attention_mask` has the same length.

In [9]:
print(tokenized['test'][0])
print(len(tokenized['test'][0]["input_ids"]))
print(len(tokenized['test'][0]["attention_mask"]))

{'label': 0, 'input_ids': [101, 1045, 2293, 16596, 1011, 10882, 1998, 2572, 5627, 2000, 2404, 2039, 2007, 1037, 2843, 1012, 16596, 1011, 10882, 5691, 1013, 2694, 2024, 2788, 2104, 11263, 25848, 1010, 2104, 1011, 12315, 1998, 28947, 1012, 1045, 2699, 2000, 2066, 2023, 1010, 1045, 2428, 2106, 1010, 2021, 2009, 2003, 2000, 2204, 2694, 16596, 1011, 10882, 2004, 17690, 1019, 2003, 2000, 2732, 10313, 1006, 1996, 2434, 1007, 1012, 10021, 4013, 3367, 20086, 2015, 1010, 10036, 19747, 4520, 1010, 25931, 3064, 22580, 1010, 1039, 2290, 2008, 2987, 1005, 1056, 2674, 1996, 4281, 1010, 1998, 16267, 2028, 1011, 8789, 3494, 3685, 2022, 9462, 2007, 1037, 1005, 16596, 1011, 10882, 1005, 4292, 1012, 1006, 1045, 1005, 1049, 2469, 2045, 2024, 2216, 1997, 2017, 2041, 2045, 2040, 2228, 17690, 1019, 2003, 2204, 16596, 1011, 102], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,

## Set the format for PyTorch

Before training, the dataset needs to return tensors instead of Python lists.
`.set_format(type="torch", columns=[...])` does this in place — it doesn't change the data, only the return type.

Set the format on the tokenized dataset so that `input_ids`, `attention_mask`, and `label` come back as tensors.

In [11]:
tokenized['test'].set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

print(tokenized['test'][0])

{'label': tensor(0), 'input_ids': tensor([  101,  1045,  2293, 16596,  1011, 10882,  1998,  2572,  5627,  2000,
         2404,  2039,  2007,  1037,  2843,  1012, 16596,  1011, 10882,  5691,
         1013,  2694,  2024,  2788,  2104, 11263, 25848,  1010,  2104,  1011,
        12315,  1998, 28947,  1012,  1045,  2699,  2000,  2066,  2023,  1010,
         1045,  2428,  2106,  1010,  2021,  2009,  2003,  2000,  2204,  2694,
        16596,  1011, 10882,  2004, 17690,  1019,  2003,  2000,  2732, 10313,
         1006,  1996,  2434,  1007,  1012, 10021,  4013,  3367, 20086,  2015,
         1010, 10036, 19747,  4520,  1010, 25931,  3064, 22580,  1010,  1039,
         2290,  2008,  2987,  1005,  1056,  2674,  1996,  4281,  1010,  1998,
        16267,  2028,  1011,  8789,  3494,  3685,  2022,  9462,  2007,  1037,
         1005, 16596,  1011, 10882,  1005,  4292,  1012,  1006,  1045,  1005,
         1049,  2469,  2045,  2024,  2216,  1997,  2017,  2041,  2045,  2040,
         2228, 17690,  1019,  

## Key takeaways

| Concept | What it means |
|---|---|
| `DatasetDict` | Container with one `Dataset` per split (train / test / validation) |
| `.filter(fn)` | Keep only rows where `fn` returns `True` — memory-efficient |
| `.map(fn, batched=True)` | Apply `fn` to every row and add the result as new columns |
| `truncation + padding` | Makes all sequences the same length so they can be batched into tensors |
| `.set_format("torch")` | Switches output type from Python lists to PyTorch tensors |
| `.push_to_hub(...)` | Uploads the dataset to your HF Hub account, versioned and shareable |